In [32]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from evaluate import load
import numpy as np


In [33]:
# 1. Load Dataset
print("Loading AG News Dataset...")
dataset = load_dataset("ag_news")


Loading AG News Dataset...


In [34]:
# 2. Tokenization
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)


In [36]:
# 4. Load Model
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=4)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [37]:
# 5. Define Metrics (Accuracy & F1)
accuracy_metric = load("accuracy")
f1_metric = load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    return {**acc, **f1}


In [38]:
training_args = TrainingArguments(
    output_dir="./bert_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [39]:
# Use DistilBERT + 10,000 samples
model_checkpoint = "distilbert-base-uncased"
train_dataset = tokenized_datasets["train"].select(range(10000))
eval_dataset = tokenized_datasets["test"].select(range(2000))

# Training with just 1 epoch
training_args = TrainingArguments(
    output_dir="./bert_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    load_best_model_at_end=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)


In [40]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from evaluate import load
import numpy as np

# 1. Define datasets
train_dataset = tokenized_datasets["train"].select(range(10000))
eval_dataset = tokenized_datasets["test"].select(range(2000))

# 2. Define Metrics
accuracy_metric = load("accuracy")
f1_metric = load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {**acc, **f1}

# 3. Load Model
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=4)

# 4. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./bert_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    load_best_model_at_end=False,
)

# 5. Initialize data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 6. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
# Start the training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.324442,0.262299,0.926000,0.925839


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.364024169921875, metrics={'train_runtime': 108.9862, 'train_samples_per_second': 91.755, 'train_steps_per_second': 11.469, 'total_flos': 220348748370240.0, 'train_loss': 0.364024169921875, 'epoch': 1.0})

In [42]:
# Evaluate the model performance
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

Evaluation results: {'eval_loss': 0.26229938864707947, 'eval_accuracy': 0.926, 'eval_f1': 0.9258387332578903, 'eval_runtime': 5.0986, 'eval_samples_per_second': 392.266, 'eval_steps_per_second': 49.033, 'epoch': 1.0}


In [43]:
import torch
print(torch.cuda.is_available())   # True = GPU available
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Running on CPU")

True
Tesla T4


In [44]:
# 8. Train & Save
print("Starting Training...")
trainer.train()


Starting Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.220436,0.296928,0.926000,0.925764


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.22273877868652345, metrics={'train_runtime': 109.9106, 'train_samples_per_second': 90.983, 'train_steps_per_second': 11.373, 'total_flos': 220348748370240.0, 'train_loss': 0.22273877868652345, 'epoch': 1.0})

In [45]:

# Save the final model
model.save_pretrained("./bert_classifier")
tokenizer.save_pretrained("./bert_classifier")
print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
